# Active-Learning Hyperspectral Invasive Species Segmentation via Adapted SAM2
## Google Colab GPU Execution Notebook

This notebook runs the complete end-to-end AL-HSI-SAM2 experiment on a free Google Colab T4 GPU (or A100/V100):
1. Environment Setup & Dependency Installation
2. Dataset & Checkpoint Fetching
3. Baseline Model Training
4. Spectral Adapter + SAM2 + LoRA Training
5. Active Learning Loop (BALD vs Entropy vs Random Queries)
6. Ablation Studies
7. Publication Plots & Visualization Dashboard Data Export

### Step 0: Verify GPU Accelerator
Ensure you have selected **Runtime > Change runtime type > T4 GPU** in the top menu.

In [ ]:
!nvidia-smi

### Step 1: Clone or Mount Project Files

**Option A (GitHub):** Replace with your GitHub repository URL if you pushed your code to GitHub:
```bash
!git clone https://github.com/your-username/AL-HSI-SAM2.git
%cd AL-HSI-SAM2
```

**Option B (Google Drive):** Mount your Google Drive where you uploaded the project folder:

In [ ]:
# Clone project repository
!git clone https://github.com/rushant22/hyperspectral-sam2-al.git
%cd hyperspectral-sam2-al

# (Alternative if using Google Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/hyperspectral-sam2-al


### Step 2: Install Dependencies
Install required scientific computing libraries and the official Meta SAM2 package.

In [ ]:
!pip install -q einops scikit-image pyyaml tqdm seaborn hydra-core omegaconf iopath
!pip install -q git+https://github.com/facebookresearch/sam2.git

### Step 3: Download Datasets & SAM 2.1 Checkpoint

In [ ]:
!python data/download.py --dataset all --sam2-checkpoint

### Step 4: Run Smoke Test
Verify that all components (models, losses, active learning loop, metrics) pass.

In [ ]:
!python scripts/smoke_test.py

### Step 5: Train Baseline Model (Without Spectral Adapter)
Establishes the performance floor on the dataset (Pavia University or Indian Pines).

In [ ]:
!python scripts/train_baseline.py --config configs/default.yaml

### Step 6: Train Adapted SAM 2 Model (With Spectral Cross-Attention Adapter + LoRA)
Trains the full architecture with separate learning rates for adapter and LoRA.

In [ ]:
!python scripts/train_adapter.py --config configs/default.yaml

### Step 7: Run Active Learning Loop
Executes multi-round active learning comparing:
- **BALD** (Bayesian Active Learning by Disagreement via MC-Dropout)
- **Shannon Entropy** (Uncertainty baseline)
- **Random Sampling** (Passive learning baseline)

In [ ]:
!python scripts/run_al_loop.py --config configs/default.yaml --strategies bald entropy random

### Step 8: Run Ablation Studies
Systematically verifies contributions of each component:
1. Number of adapter queries (M = 4, 8, 12, 16)
2. LoRA rank (r = 0, 4, 8, 16)
3. PCA residual branch (w/ vs w/o)
4. MC-Dropout passes (T = 5, 10, 20)

In [ ]:
!python scripts/run_ablations.py --config configs/default.yaml --ablation all

### Step 9: Zip & Download Results
Packages all logs, trained model weights, evaluation plots, and dashboard data into a zip file for download.

In [ ]:
!zip -r experiment_results.zip results/ paper/figures/
from google.colab import files
files.download('experiment_results.zip')